# 09 · Structured Credit & Tranches

Structured credit deals (CLO/ABS/RMBS) are described via JSON payloads containing pools, tranches, and waterfalls. Finstack parses the spec and prices the structure in one step.

### Learning Objectives
- Construct a minimal structured-credit JSON payload (assets, tranches, waterfall)
- Load it with `StructuredCredit.from_json` and price via the standard registry
- Request metrics such as par spread and expected loss for mezzanine tranches

In [ ]:
import json
from datetime import date

from finstack.core.market_data import MarketContext
from finstack.core.market_data.term_structures import DiscountCurve
from finstack.valuations.instruments import StructuredCredit
from finstack.valuations.pricer import create_standard_registry

as_of = date(2024, 1, 2)
market = MarketContext()
market.insert_discount(
    DiscountCurve(
        "USD-OIS",
        as_of,
        [(0.0, 1.0), (1.0, 0.9960), (3.0, 0.9820), (5.0, 0.9550), (10.0, 0.9050)],
    )
)
registry = create_standard_registry()

CURRENCY = "USD"

def money(amount):
    return {"amount": amount, "currency": CURRENCY}


def asset_entry(identifier, balance, rate, maturity, *, kind):
    return {
        "id": identifier,
        "asset_type": kind,
        "balance": money(balance),
        "rate": rate,
        "maturity": maturity,
        "credit_quality": "BBB",
        "obligor_id": f"OBL-{identifier}",
        "is_defaulted": False,
    }


def tranche_entry(identifier, attach, detach, rate, balance, seniority, priority):
    return {
        "id": identifier,
        "attachment_point": attach,
        "detachment_point": detach,
        "seniority": seniority,
        "rating": "BBB" if seniority != "equity" else "NR",
        "original_balance": money(balance),
        "current_balance": money(balance),
        "coupon": {"Fixed": {"rate": rate}},
        "payment_frequency": {"Months": 3},
        "day_count": "Act360",
        "payment_priority": priority,
        "attributes": {"tags": [], "meta": {}},
    }


def waterfall_tiers(tranche_ids):
    tiers = []
    tiers.append(
        {
            "id": "fees",
            "priority": 1,
            "recipients": [
                {
                    "id": "trustee_fee",
                    "recipient_type": {"ServiceProvider": "Trustee"},
                    "calculation": {"FixedAmount": {"amount": money(25_000.0)}},
                }
            ],
            "payment_type": "Fee",
            "allocation_mode": "Sequential",
        }
    )
    priority = 2
    for tid in tranche_ids:
        tiers.append(
            {
                "id": f"{tid}_interest",
                "priority": priority,
                "recipients": [
                    {
                        "id": f"{tid}_int",
                        "recipient_type": {"Tranche": tid},
                        "calculation": {"TrancheInterest": {"tranche_id": tid}},
                    }
                ],
                "payment_type": "Interest",
                "allocation_mode": "Sequential",
            }
        )
        priority += 1
    for tid in tranche_ids:
        tiers.append(
            {
                "id": f"{tid}_principal",
                "priority": priority,
                "recipients": [
                    {
                        "id": f"{tid}_prin",
                        "recipient_type": {"Tranche": tid},
                        "calculation": {"TranchePrincipal": {"tranche_id": tid}},
                    }
                ],
                "payment_type": "Principal",
                "allocation_mode": "Sequential",
            }
        )
        priority += 1
    tiers.append(
        {
            "id": "equity_residual",
            "priority": priority,
            "recipients": [
                {"id": "equity", "recipient_type": "Equity", "calculation": "ResidualCash"}
            ],
            "payment_type": "Residual",
            "allocation_mode": "Sequential",
        }
    )
    return {"tiers": tiers, "base_currency": CURRENCY}


def build_simple_structured_credit():
    assets = [
        asset_entry("ASSET1", 6_000_000.0, 0.072, "2032-01-01", kind={"type": "FirstLienLoan"}),
        asset_entry("ASSET2", 4_000_000.0, 0.068, "2031-06-01", kind={"type": "FirstLienLoan"}),
    ]
    tranches = [
        tranche_entry("SENIOR", 0.0, 0.60, 0.035, 7_000_000.0, "Senior", 1),
        tranche_entry("MEZZ", 0.60, 0.90, 0.065, 2_500_000.0, "Mezzanine", 2),
        tranche_entry("EQUITY", 0.90, 1.00, 0.0, 500_000.0, "equity", 3),
    ]
    deal = {
        "id": "CLO-DEMO",
        "currency": CURRENCY,
        "discount_curve_id": "USD-OIS",
        "start_date": "2024-01-02",
        "waterfall": waterfall_tiers([t["id"] for t in tranches[:-1]]),
        "assets": assets,
        "tranches": tranches,
    }
    return json.dumps(deal)

spec_json = build_simple_structured_credit()
structured_credit = StructuredCredit.from_json(spec_json)


## Price the Deal
The structured-credit pricer consumes the JSON spec, runs the waterfall, and returns PV plus requested metrics.

In [ ]:
result = registry.price_with_metrics(
    structured_credit,
    "discounting",
    market,
    ["par_spread", "expected_loss"],
)
print("PV:", result.value)
print("Par spread:", result.measures.get("par_spread"))
print("Expected loss:", result.measures.get("expected_loss"))


## Summary
- Structured credit deals are provided as JSON documents referencing discount-curve IDs already loaded in `MarketContext`.
- Minimal specs include asset definitions, tranche stack, and a waterfall describing fee/interest/principal ordering.
- `StructuredCredit.from_json` plus `price_with_metrics` produces PV, spreads, expected loss, and other reporting measures without additional glue code.